# HOG + SVM/KNN cho Kaggle Flowers 7 lớp

Dataset sử dụng: https://www.kaggle.com/datasets/nadyana/flowers

Notebook này đọc ảnh trực tiếp từ các thư mục lớp, trích xuất đặc trưng HOG, chia train/test 80/20, huấn luyện SVM và KNN, đánh giá Accuracy/Precision/Recall/F1-score, rồi lưu biểu đồ PNG vào `results/`.

## 1. Import thư viện và cấu hình

Nếu dataset đã được giải nén trong project, notebook sẽ tự tìm trong các thư mục phổ biến như `flowers/`, `flower-training/`, `dataset/`, `data/`. Nếu dataset nằm ngoài project, hãy sửa biến `DATASET_ROOT_OVERRIDE` ở cell dưới.

In [ ]:
from pathlib import Path
import os
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from skimage.feature import hog
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, learning_curve, train_test_split
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Nếu dataset Kaggle nằm ngoài project, sửa dòng này, ví dụ:
# DATASET_ROOT_OVERRIDE = r"C:\\Users\\OS\\Downloads\\flowers"
DATASET_ROOT_OVERRIDE = None

CLASS_NAMES = ["Bellflower", "Daisy", "Dandelion", "Lotus", "Rose", "Sunflower", "Tulips"]
CLASS_KEYS = ["bellflower", "daisy", "dandelion", "lotus", "rose", "sunflower", "tulips"]
CLASS_ALIASES = {
    "bellflower": ["bellflower", "bellflowers"],
    "daisy": ["daisy", "daisies"],
    "dandelion": ["dandelion", "dandelions"],
    "lotus": ["lotus"],
    "rose": ["rose", "roses"],
    "sunflower": ["sunflower", "sunflowers"],
    "tulips": ["tulip", "tulips"],
}

IMG_SIZE = (128, 128)
TEST_SIZE = 0.20
RANDOM_STATE = 42
EXPECTED_IMAGES_PER_CLASS = 1600

# Learning curve train nhiều lần nên dùng tập con stratified từ dữ liệu thật để chạy thực tế hơn.
# Metrics chính và confusion matrix vẫn train/evaluate trên toàn bộ train/test 80/20.
LEARNING_CURVE_MAX_SAMPLES = 3500

print("Project root:", PROJECT_ROOT)
print("Results dir:", RESULTS_DIR)

## 2. Tự động tìm dataset và đọc ảnh

Cell này tìm thư mục chứa đủ 7 lớp. Tên thư mục có thể viết hoa/thường, và lớp Tulips có thể là `tulip` hoặc `tulips`.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def normalize_name(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def resolve_class_dirs(root: Path) -> dict[str, Path] | None:
    if not root.exists() or not root.is_dir():
        return None

    child_dirs = [p for p in root.iterdir() if p.is_dir()]
    by_name = {normalize_name(p.name): p for p in child_dirs}
    resolved = {}

    for key in CLASS_KEYS:
        found = None
        for alias in CLASS_ALIASES[key]:
            alias_key = normalize_name(alias)
            if alias_key in by_name:
                found = by_name[alias_key]
                break
        if found is None:
            return None
        resolved[key] = found

    return resolved


def candidate_dataset_roots(project_root: Path) -> list[Path]:
    candidates = []

    if DATASET_ROOT_OVERRIDE:
        candidates.append(Path(DATASET_ROOT_OVERRIDE))

    env_path = os.environ.get("FLOWERS_DATASET_DIR")
    if env_path:
        candidates.append(Path(env_path))

    common_names = [
        "flowers",
        "flower",
        "flower-training",
        "flowers-training",
        "dataset",
        "data",
        "kaggle-flower-dataset",
        "preprocessed_outputs",
    ]
    candidates.extend(project_root / name for name in common_names)

    # Tìm thêm trong project tối đa 3 cấp để bắt các cấu trúc như dataset/flowers/<class>.
    for path in project_root.rglob("*"):
        if path.is_dir() and len(path.relative_to(project_root).parts) <= 3:
            candidates.append(path)

    unique = []
    seen = set()
    for path in candidates:
        resolved = path.resolve()
        if resolved not in seen:
            unique.append(path)
            seen.add(resolved)
    return unique


def count_images(class_dirs: dict[str, Path]) -> dict[str, int]:
    counts = {}
    for key, folder in class_dirs.items():
        counts[key] = sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    return counts


valid_candidates = []
for candidate in candidate_dataset_roots(PROJECT_ROOT):
    class_dirs = resolve_class_dirs(candidate)
    if class_dirs is not None:
        counts = count_images(class_dirs)
        total = sum(counts.values())
        valid_candidates.append((total, candidate, class_dirs, counts))

if not valid_candidates:
    raise FileNotFoundError(
        "Không tìm thấy thư mục dataset có đủ 7 lớp. "
        "Hãy giải nén Kaggle dataset vào project hoặc đặt DATASET_ROOT_OVERRIDE/FLOWERS_DATASET_DIR."
    )

valid_candidates.sort(key=lambda item: item[0], reverse=True)
total_images, DATASET_DIR, CLASS_DIRS, class_counts = valid_candidates[0]

print("Dataset đang dùng:", DATASET_DIR)
print("Số ảnh từng lớp:")
for display_name, key in zip(CLASS_NAMES, CLASS_KEYS):
    print(f"{display_name:10s}: {class_counts[key]}")
print("Tổng số ảnh:", total_images)

if any(count < EXPECTED_IMAGES_PER_CLASS for count in class_counts.values()):
    print("\nCẢNH BÁO: Dataset tìm được chưa đủ 1600 ảnh/lớp như Kaggle nadyana/flowers.")
    print("Nếu muốn kết quả lớn đúng như báo cáo, hãy trỏ DATASET_ROOT_OVERRIDE tới thư mục Kaggle đầy đủ.")

## 3. Trích xuất đặc trưng HOG

Ảnh được resize về `128 x 128`, chuyển grayscale, sau đó trích xuất HOG bằng `skimage.feature.hog`.

In [ ]:
def collect_image_paths(class_dirs: dict[str, Path]) -> tuple[list[Path], np.ndarray]:
    paths = []
    labels = []
    for label, key in enumerate(CLASS_KEYS):
        cls_paths = sorted(
            p for p in class_dirs[key].rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        paths.extend(cls_paths)
        labels.extend([label] * len(cls_paths))
    return paths, np.asarray(labels, dtype=np.int64)


def extract_hog_feature(image_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Không đọc được ảnh: {image_path}")
    image = cv2.resize(image, IMG_SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    feature = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    )
    return feature.astype(np.float32)


image_paths, y = collect_image_paths(CLASS_DIRS)
X_features = []
failed_images = []

for idx, path in enumerate(image_paths, start=1):
    try:
        X_features.append(extract_hog_feature(path))
    except Exception as exc:
        failed_images.append((path, str(exc)))
    if idx % 500 == 0 or idx == len(image_paths):
        print(f"Đã xử lý {idx}/{len(image_paths)} ảnh")

if failed_images:
    failed_set = {path for path, _ in failed_images}
    keep_indices = [i for i, path in enumerate(image_paths) if path not in failed_set]
    y = y[keep_indices]

X = np.asarray(X_features, dtype=np.float32)

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)
print("Số ảnh lỗi:", len(failed_images))

## 4. Chia train/test 80/20

Dữ liệu được chia stratified để giữ tỷ lệ lớp trong train/test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train theo lớp:", np.bincount(y_train, minlength=len(CLASS_NAMES)))
print("Test theo lớp:", np.bincount(y_test, minlength=len(CLASS_NAMES)))

## 5. Huấn luyện SVM và KNN

SVM dùng `LinearSVC(C=0.003)` sau khi tune validation để giảm overfit. KNN dùng `n_neighbors=3`, `weights="uniform"`, `metric="manhattan"` để tránh train accuracy 1.0 do mỗi ảnh train tự vote cho chính nó.

In [ ]:
models = {
    "HOG + SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LinearSVC(
            C=0.003,
            class_weight="balanced",
            dual="auto",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "HOG + KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="manhattan", n_jobs=-1)),
    ]),
}

predictions = {}
for name, model in models.items():
    print("Đang huấn luyện:", name)
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    print("Hoàn tất:", name)

## 6. Đánh giá Accuracy, Precision, Recall, F1-score

Precision/Recall/F1 được tính theo macro average để các lớp có trọng số ngang nhau.

In [ ]:
metrics_summary = []

for name, y_pred in predictions.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0,
    )
    metrics_summary.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
    })

    print("=" * 80)
    print(name)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

metrics_summary

## 7. Vẽ và lưu Confusion Matrix

Mỗi mô hình có một file PNG riêng trong `results/`.

In [ ]:
def plot_confusion_matrix_png(y_true, y_pred, title: str, output_name: str) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
    output_path = RESULTS_DIR / output_name

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cbar=True,
    )
    plt.title(title, fontsize=15, weight="bold")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", output_path)


plot_confusion_matrix_png(y_test, predictions["HOG + SVM"], "Confusion Matrix - HOG + SVM", "hog_svm_flowers_confusion_matrix.png")
plot_confusion_matrix_png(y_test, predictions["HOG + KNN"], "Confusion Matrix - HOG + KNN", "hog_knn_flowers_confusion_matrix.png")

## 8. Vẽ và lưu Learning Curve

Learning curve được tính bằng cross-validation trên đặc trưng HOG thật. Nếu máy yếu, có thể đặt `LEARNING_CURVE_MAX_SAMPLES = 5000` ở cell cấu hình để dùng một tập con stratified từ dữ liệu thật.

In [ ]:
def get_learning_curve_data():
    if LEARNING_CURVE_MAX_SAMPLES is None or LEARNING_CURVE_MAX_SAMPLES >= len(y):
        return X, y

    _, X_lc, _, y_lc = train_test_split(
        X,
        y,
        test_size=LEARNING_CURVE_MAX_SAMPLES,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    return X_lc, y_lc


def plot_learning_curve_png(model, title: str, output_name: str) -> None:
    X_lc, y_lc = get_learning_curve_data()
    min_class_count = int(np.bincount(y_lc, minlength=len(CLASS_NAMES)).min())
    n_splits = min(3, min_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    train_sizes, train_scores, valid_scores = learning_curve(
        model,
        X_lc,
        y_lc,
        cv=cv,
        train_sizes=np.linspace(0.2, 1.0, 5),
        scoring="accuracy",
        n_jobs=1,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    valid_mean = valid_scores.mean(axis=1)
    valid_std = valid_scores.std(axis=1)
    output_path = RESULTS_DIR / output_name

    plt.figure(figsize=(8.5, 6))
    plt.plot(train_sizes, train_mean, marker="o", linewidth=2, label="Training score")
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.18)
    plt.plot(train_sizes, valid_mean, marker="s", linewidth=2, label="Validation score")
    plt.fill_between(train_sizes, valid_mean - valid_std, valid_mean + valid_std, alpha=0.18)
    plt.title(title, fontsize=15, weight="bold")
    plt.xlabel("Training set size")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", output_path)


plot_learning_curve_png(models["HOG + SVM"], "Learning Curve - HOG + SVM", "hog_svm_flowers_learning_curve.png")
plot_learning_curve_png(models["HOG + KNN"], "Learning Curve - HOG + KNN", "hog_knn_flowers_learning_curve.png")

## 9. Accuracy Comparison Chart

Biểu đồ so sánh accuracy trên tập test giữa SVM và KNN.

In [ ]:
model_names = [row["Model"] for row in metrics_summary]
accuracies = [row["Accuracy"] for row in metrics_summary]
output_path = RESULTS_DIR / "hog_svm_knn_flowers_accuracy_comparison.png"

plt.figure(figsize=(7.5, 5.5))
ax = sns.barplot(x=model_names, y=accuracies, palette=["#3f8cff", "#20b486"])
plt.title("Accuracy Comparison - HOG + SVM vs HOG + KNN", fontsize=15, weight="bold")
plt.xlabel("Model")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1.05)

for bar, acc in zip(ax.patches, accuracies):
    ax.annotate(
        f"{acc:.4f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center",
        va="bottom",
        fontsize=11,
        weight="bold",
        xytext=(0, 5),
        textcoords="offset points",
    )

plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
print("Đã lưu:", output_path)

## 10. Danh sách file kết quả

Cell này in ra các file PNG đã được sinh trong thư mục `results/`.

In [ ]:
for path in sorted(RESULTS_DIR.glob("hog_*flowers*.png")):
    print(path.name)